# 05 — Phenology Overlap (Full Scale)

Computes the temporal overlap scalar Δ for all pairs in the shared
pair universe using the full corrected datasets.

Δ = Σ_t min(f̃_t, ã_t) — coefficient of overlapping (Ridout & Linkie 2009)

where f̃ and ã are the normalized 52-week flowering and activity curves.

This notebook computes Δ at the pair level (averaged across shared bins)
as used in ANTHEIA-Scalar. The spatiotemporal embeddings V_δ (4D and 15D)
used in ANTHEIA-4D and ANTHEIA-15D are computed in the representation
notebooks from the PPE opportunity surface directly.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import gc

BASE         = Path("/scratch/ariana.l")
F_CURVES_IN  = BASE / "New Stage 4 Link Prediction Model" / "f_curves_ppe.csv"
A_CURVES_IN  = BASE / "New Stage 4 Link Prediction Model" / "a_curves_corrected.csv"
GLOBIOUT     = BASE / "New Stage 4 Link Prediction Model" / "globiinteractions_conus.csv"
OUT_DIR      = BASE / "New Stage 4 Link Prediction Model"

print("Paths OK")

In [ ]:
# Load f_curves and a_curves
print("Loading f_curves (plant flowering, PPE-derived)...")
f_curves = pd.read_csv(F_CURVES_IN, index_col=0)
print(f"  f_curves shape: {f_curves.shape}")

print("Loading a_curves (pollinator activity, GBIF-derived)...")
a_curves = pd.read_csv(A_CURVES_IN, index_col=0)
print(f"  a_curves shape: {a_curves.shape}")
print()
print("Note: a_curves are GBIF observation histograms — they reflect recorder")
print("effort, not true pollinator phenology. SDM-derived curves replace these")
print("in the SDM comparison experiments.")

In [ ]:
# Load GloBI interaction pairs
globidf = pd.read_csv(GLOBIOUT)
pairs = globidf[['plant_species', 'pollinator_species']].drop_duplicates()

# Filter to species present in both f_curves and a_curves
f_species = set(f_curves.index)
a_species = set(a_curves.index)

pairs = pairs[
    pairs['plant_species'].isin(f_species) &
    pairs['pollinator_species'].isin(a_species)
].reset_index(drop=True)

print(f"Pairs with coverage in both f_curves and a_curves: {len(pairs)}")

In [ ]:
# Compute Δ for each pair
# Δ = Σ_t min(f̃_t, ã_t)
# Both curves are already normalized to sum = 1

print("Computing Δ for all pairs...")
deltas = []
for _, row in pairs.iterrows():
    plant = row['plant_species']
    pollinator = row['pollinator_species']
    f = f_curves.loc[plant].values
    a = a_curves.loc[pollinator].values
    delta = np.minimum(f, a).sum()
    deltas.append({'plant_species': plant, 'pollinator_species': pollinator, 'delta': delta})

delta_df = pd.DataFrame(deltas)
print(f"Δ distribution:")
print(delta_df['delta'].describe())

In [ ]:
# Save
out_path = OUT_DIR / 'delta_overlap_pairs.csv'
delta_df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
delta_df.head(10)